In [ ]:
# from typing import Literal
# from langchain.chat_models import init_chat_model
# from langchain.tools import tool
# from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
# from langgraph.constants import START, END
# from langgraph.graph import StateGraph, MessagesState
#
# # ====== Step 1: 准备工具 ======
# # 工具
# @tool
# def get_weather(city: str) -> str:
#     """获取城市天气"""
#     weather_data = {
#         "北京": "晴天 25度",
#         "上海": "多云 28度",
#         "杭州": "小雨 22度",
#     }
#     return weather_data.get(city, f"未找到{city}的天气信息")
#
# @tool
# def square_root(x: float) -> str:
#     """Calculate the square root of a number"""
#     return x ** 0.5
#
# # 组成工具数组
# tools = [get_weather, square_root]
# # 组成工具名和工具的映射，方便查找工具
# tools_by_name = {t.name: t for t in tools}
#
# # ====== Step 2: 准备模型，绑定工具 ======
# # 定义模型
# model = init_chat_model(
#     'deepseek-v4-flash',
#     extra_body = {'thinking': {'type': 'disabled'}}
# )
# # 绑定工具到模型，与create_agent类似，将来LangChain会自动帮我们拼接tool信息发给模型
# model_with_tools = model.bind_tools(tools)
#
#
# # ====== Step 3: 定义Node ======
# def tool_node(state: MessagesState):
#     # 获取最后一条消息，也就是AI返回的带工具的消息
#     message = state['messages'][-1]
#     messages=[]
#     # 获取工具, 逐个处理
#     for tool_call in message.tool_calls:
#         # 获取工具信息
#         _name = tool_call['name']
#         _args = tool_call['args']
#         _id = tool_call['id']
#         # 调用工具
#         result = tools_by_name[_name].invoke(_args)
#         # 封装ToolMessage
#         messages.append(ToolMessage(content=result, tool_call_id = _id))
#     # 返回结果
#     return {'messages': messages}
#
# def llm_node(state: MessagesState):
#     """LLM节点：调用模型，可以选择调用工具"""
#     response = model_with_tools.invoke(state["messages"])
#     return {"messages": [response]}
#
# # 要定义路由，就是看AI返回消息中是否带有tool_call，有就走tool_node，没有就走END
# def should_continue(state: MessagesState) -> Literal["tools", END]:
#     """路由函数：判断下一步是继续调工具还是结束"""
#     last_message = state["messages"][-1]
#     # 如果包含tool_calls，则跳转到tools节点，没有则跳转到END节点
#     if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
#         return "tools"  # 有工具调用 -> 去工具节点
#     return END
#
# # ====== Step 4: 构建图 ======
# builder = StateGraph(MessagesState)
# builder.add_node("llm", llm_node)
# builder.add_node("tools", tool_node)
# builder.add_edge(START, "llm")
# builder.add_conditional_edges("llm", should_continue)
# builder.add_edge("tools", "llm")  # 工具执行后回到LLM，形成循环
# agent_graph = builder.compile()
#
# # ====== Step 5: 调用 ======
# result = agent_graph.invoke({
#     "messages": [HumanMessage(content="北京和杭州今天天气怎么样？")]
# })
# for m in result['messages']:
#     m.pretty_print()

In [ ]:
# from langchain.chat_models import init_chat_model
# from langchain.tools import tool
# from langchain_core.messages import HumanMessage
# from langgraph.constants import START, END
# from langgraph.graph import StateGraph, MessagesState
# from langgraph.prebuilt import ToolNode, tools_condition
#
#
# # ====== Step 1: 准备工具 ======
# # 工具
# @tool
# def get_weather(city: str) -> str:
#     """获取城市天气"""
#     weather_data = {
#         "北京": "晴天 25度",
#         "上海": "多云 28度",
#         "杭州": "小雨 22度",
#     }
#     return weather_data.get(city, f"未找到{city}的天气信息")
#
# # 组成工具数组
# tools = [get_weather]
#
# # ====== Step 2: 准备模型，绑定工具 ======
# # 定义模型
# model = init_chat_model(
#     'deepseek-v4-flash',
#     extra_body = {'thinking': {'type': 'disabled'}}
# )
# # 绑定工具到模型，与create_agent类似，将来LangChain会自动帮我们拼接tool信息发给模型
# model_with_tools = model.bind_tools(tools)
#
#
# # ====== Step 3: 定义Node ======
# def llm_node(state: MessagesState):
#     """LLM节点：调用模型，可以选择调用工具"""
#     response = model_with_tools.invoke(state["messages"])
#     return {"messages": [response]}
#
# # ====== Step 4: 构建图 ======
# builder = StateGraph(MessagesState)
# builder.add_node("llm", llm_node)
# #读最后一条 AIMessage 的 tool_calls → 找到同名工具执行 → 把结果包装成 ToolMessage 追加到 messages
# builder.add_node("tools", ToolNode(tools))
# builder.add_edge(START, "llm")
# #看最后一条 AIMessage 有没有 tool_calls —— 有就返回 "tools"，没有就返回 END。
# builder.add_conditional_edges("llm", tools_condition)
# builder.add_edge("tools", "llm")  # 工具执行后回到LLM，形成循环
# agent_graph = builder.compile()
#
# # ====== Step 5: 调用 ======
# result = agent_graph.invoke({
#     "messages": [HumanMessage(content="北京和杭州今天天气怎么样？")]
# })
# for m in result['messages']:
#     m.pretty_print()

In [ ]:
#################################### 输出图 #########################################

# # 1. Mermaid 源码（本地，无需联网）
# print(agent_graph.get_graph().draw_mermaid())
#
# # 2. 渲染 PNG 并保存（需要联网，走 mermaid.ink）
# png_data = agent_graph.get_graph().draw_mermaid_png()
# with open("tool_graph.png", "wb") as f:
#     f.write(png_data)
#
# print("图表已成功保存为 tool_graph.png")

In [4]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.constants import START, END
from langgraph.graph import StateGraph, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from typing import TypedDict

from typing_extensions import Annotated


@tool
def tool_weather(region: str) -> str:
    """
    根据地区查询天气
    Args:
        region: 地区名称，例如"北京""上海"。
    Returns:
        该地区的天气情况文本。
    """
    return f"{region}：晴天，25度"

#llm
llm = init_chat_model(
    model = "deepseek-flash",
    extra_body = {'thinking': {'type': 'disabled'}}
)

#llm绑定工具
tools = [tool_weather]
llm_tool = llm.bind_tools(tools)

#State用langGraph提供的MessagesState即可
#定义node
def llm_node(state:MessagesState):
    ai_message = llm_tool.invoke(state["messages"])
    # MessagesState 用的是 add_messages reducer，只需返回新增的消息，不能再手动 append
    return {"messages": [ai_message]}

#构建图
builder_graph = StateGraph(MessagesState)
##添加node和Edge
builder_graph.add_node("llm_node",llm_node)
builder_graph.add_node("tools",ToolNode(tools))
builder_graph.add_edge(START,"llm_node")
builder_graph.add_conditional_edges("llm_node",tools_condition)
builder_graph.add_edge("tools","llm_node")   # 工具执行后回到模型，形成循环

#编译图和执行图
graph = builder_graph.compile()
result = graph.invoke({
    "messages": [
        HumanMessage(content="北京天气怎么样？")
    ]
})
print(result)

{'messages': [HumanMessage(content='北京天气怎么样？', additional_kwargs={}, response_metadata={}, id='bfa73dd7-b93a-48bb-bf9b-21eab182c0c4'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 298, 'total_tokens': 336, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 128, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 128, 'prompt_cache_miss_tokens': 170}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': 'df61b026-b443-4380-bc03-f0374e50e2e7', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1da13-8fb8-7220-95a4-7a74e2ff04c9-0', tool_calls=[{'name': 'tool_weather', 'args': {'region': '北京'}, 'id': 'call_00_HruvHQbwa5rdkTdjSGz58844', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 298, 'ou